---
# Phase 1 — Data Loading & Preprocessing

## 0. Install

In [1]:
!pip install ucimlrepo imbalanced-learn groq transformers torch tqdm umap-learn shap xgboost --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.2 MB/s eta 0:00:00


## 1. Imports & global config

All libraries for the entire pipeline imported here once.

In [2]:
import os, json, time, warnings, pickle, shutil, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm.auto import tqdm

from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    average_precision_score, roc_curve, precision_recall_curve,
    confusion_matrix, ConfusionMatrixDisplay,
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.calibration import calibration_curve
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from groq import Groq, RateLimitError

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
import shap

warnings.filterwarnings("ignore")
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

plt.rcParams.update({
    "figure.facecolor":"white","axes.facecolor":"#F8F9FB",
    "axes.grid":True,"grid.alpha":0.4,
    "axes.spines.top":False,"axes.spines.right":False,"font.size":11,
})
PALETTE = {
    "neg":"#4A90D9","pos":"#E05C5C","neu":"#7F77DD",
    "lr":"#4A90D9","rf":"#56B87A","xgb":"#E0AA00",
    "early":"#E05C5C","late":"#9B59B6","tab":"#7F77DD",
}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs("data", exist_ok=True)
os.makedirs("data/models", exist_ok=True)
os.makedirs("data/figures", exist_ok=True)
os.makedirs("data/paper", exist_ok=True)
print(f"Ready. RANDOM_SEED={RANDOM_SEED} | device={device}")

KeyboardInterrupt: 

## 2. Google Drive — mount & restore

Run at the start of **every new Colab session** to restore saved artefacts.

In [4]:
from google.colab import drive, userdata
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/diabetes_project/data"
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/models", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/paper", exist_ok=True)

RESTORE_FILES = [
    "scaler.pkl","feature_names.json","diabetes_clean.parquet",
    "X_train.npy","X_val.npy","X_test.npy",
    "y_train.npy","y_val.npy","y_test.npy",
    "text_embeddings.npy","synthetic_notes_final.csv",
    "synthetic_notes.csv","notes_checkpoint.csv",
    "results_summary.csv","shap_values_xgb.npy",
    "recommendations_sample.csv",
]
for fname in RESTORE_FILES:
    src, dst = f"{DRIVE_DIR}/{fname}", f"data/{fname}"
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst); print(f"  Restored: {fname}")

for mname in ["xgboost.json","logistic_regression.pkl",
              "random_forest.pkl","early_fusion.pt","late_fusion.pt"]:
    src, dst = f"{DRIVE_DIR}/models/{mname}", f"data/models/{mname}"
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst); print(f"Restored model: {mname}")

print("Restore complete.")

Mounted at /content/drive


FileNotFoundError: [Errno 2] No such file or directory: 'data/scaler.pkl'

## 3. Load dataset

101,766 hospital encounters, 130 US hospitals, 1999–2008.

In [ ]:
raw = fetch_ucirepo(id=296)
X = raw.data.features.copy()
y = raw.data.targets.copy()
df = pd.concat([X, y], axis=1)
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Memory: {df.memory_usage(deep=True).sum()/1e6:.1f} MB")
df.head(3)

## 4. Define binary target

- **1** = readmitted <30 days (high risk)
- **0** = >30 days or NO (low risk)

Reference: Strack et al. (2014)

In [ ]:
print("Original distribution:")
print(df["readmitted"].value_counts())

df["target"] = (df["readmitted"] == "<30").astype(int)
n_pos = df["target"].sum()
n_neg = len(df) - n_pos
ratio = n_pos / len(df)
print(f"\nHigh risk (1): {n_pos:,} ({ratio:.1%})")
print(f"Low  risk (0): {n_neg:,} ({1-ratio:.1%})")
print("NOTE: ~11% positive — addressed with SMOTE on train only")

## 5. Feature selection

Four semantic groups (used later for SHAP interpretation).

In [ ]:
DEMOGRAPHIC = [
    "race","gender","age","admission_type_id",
    "discharge_disposition_id","admission_source_id","time_in_hospital",
]
CLINICAL = [
    "num_lab_procedures","num_procedures","num_medications",
    "number_outpatient","number_emergency","number_inpatient","number_diagnoses",
]
DIABETES_SPECIFIC = [
    "max_glu_serum","A1Cresult","insulin","metformin",
    "glipizide","glyburide","pioglitazone","rosiglitazone",
    "change","diabetesMed",
]
DIAGNOSIS = ["diag_1","diag_2","diag_3"]

SELECTED_FEATURES = DEMOGRAPHIC + CLINICAL + DIABETES_SPECIFIC + DIAGNOSIS
df_sel = df[SELECTED_FEATURES + ["target"]].copy()
print(f"Selected {len(SELECTED_FEATURES)} / {X.shape[1]} features")

## 6. Missing values

Dataset encodes missing as `"?"`. Strategy: drop >40%, median/mode impute rest.

In [ ]:
df_sel.replace("?", np.nan, inplace=True)

missing = df_sel.isnull().sum()
missing_pct = (missing/len(df_sel)*100).round(2)
missing_df = pd.DataFrame({"count":missing,"pct":missing_pct})
missing_df = missing_df[missing_df["count"]>0].sort_values("pct",ascending=False)
print(missing_df.to_string())

DROP_THRESHOLD = 40.0
cols_to_drop = missing_df[missing_df["pct"]>DROP_THRESHOLD].index.tolist()
if cols_to_drop:
    print(f"Dropping: {cols_to_drop}")
    df_sel.drop(columns=cols_to_drop, inplace=True)

NUM_COLS = [c for c in df_sel.select_dtypes(include=["int64","float64"]).columns if c!="target"]
CAT_COLS = [c for c in df_sel.select_dtypes(include="object").columns if c!="target"]
for col in NUM_COLS:
    if df_sel[col].isnull().any(): df_sel[col].fillna(df_sel[col].median(), inplace=True)
for col in CAT_COLS:
    if df_sel[col].isnull().any(): df_sel[col].fillna(df_sel[col].mode()[0], inplace=True)

print(f"Remaining missing: {df_sel.isnull().sum().sum()}")

## 7. ICD-9 grouping

700+ codes → 9 clinical categories (Strack et al., 2014, Table 2).

In [ ]:
def icd9_to_category(code):
    if pd.isna(code): return "Other"
    code = str(code)
    if code.startswith("E") or code.startswith("V"): return "Other"
    try: c = float(code)
    except ValueError: return "Other"
    if   390<=c<=459 or c==785: return "Circulatory"
    elif 460<=c<=519 or c==786: return "Respiratory"
    elif 520<=c<=579 or c==787: return "Digestive"
    elif c==250: return "Diabetes"
    elif 800<=c<=999: return "Injury"
    elif 710<=c<=739: return "Musculoskeletal"
    elif 580<=c<=629 or c==788: return "Genitourinary"
    elif 140<=c<=239: return "Neoplasms"
    else: return "Other"

for col in ["diag_1","diag_2","diag_3"]:
    if col in df_sel.columns:
        df_sel[col] = df_sel[col].apply(icd9_to_category)
print("ICD-9 grouping complete.")

## 8. Encoding

In [ ]:
AGE_MAP = {"[0-10)":0,"[10-20)":1,"[20-30)":2,"[30-40)":3,"[40-50)":4,
           "[50-60)":5,"[60-70)":6,"[70-80)":7,"[80-90)":8,"[90-100)":9}
if "age" in df_sel.columns: df_sel["age"] = df_sel["age"].map(AGE_MAP)
if "gender" in df_sel.columns:
    df_sel["gender"] = df_sel["gender"].map({"Male":1,"Female":0,"Unknown/Invalid":np.nan})
    df_sel["gender"].fillna(df_sel["gender"].mode()[0], inplace=True)
if "A1Cresult"     in df_sel.columns: df_sel["A1Cresult"]     = df_sel["A1Cresult"].map({"None":0,"Norm":1,">7":2,">8":3}).fillna(0).astype(int)
if "max_glu_serum" in df_sel.columns: df_sel["max_glu_serum"] = df_sel["max_glu_serum"].map({"None":0,"Norm":1,">200":2,">300":3}).fillna(0).astype(int)

MED_MAP = {"No":0,"Steady":1,"Down":2,"Up":2}
for col in ["metformin","glipizide","glyburide","pioglitazone","rosiglitazone","insulin"]:
    if col in df_sel.columns: df_sel[col] = df_sel[col].map(MED_MAP).fillna(0).astype(int)
if "change"      in df_sel.columns: df_sel["change"]      = df_sel["change"].map({"No":0,"Ch":1}).fillna(0).astype(int)
if "diabetesMed" in df_sel.columns: df_sel["diabetesMed"] = df_sel["diabetesMed"].map({"No":0,"Yes":1}).fillna(0).astype(int)

remaining_cat = [c for c in df_sel.select_dtypes(include="object").columns if c!="target"]
if remaining_cat: df_sel = pd.get_dummies(df_sel, columns=remaining_cat, drop_first=True)

bool_cols = df_sel.select_dtypes(include="bool").columns
if len(bool_cols): df_sel[bool_cols] = df_sel[bool_cols].astype(int)

before = len(df_sel)
df_sel.drop_duplicates(inplace=True)
print(f"Duplicates removed: {before-len(df_sel)}")
print(f"Shape: {df_sel.shape}")

## 9. EDA — distributions & correlation

In [ ]:
KEY_NUM = [c for c in ["time_in_hospital","num_medications","num_lab_procedures",
                        "number_inpatient","number_diagnoses","num_procedures"] if c in df_sel.columns]
fig, axes = plt.subplots(2,3,figsize=(14,8)); axes=axes.flatten()
for i,col in enumerate(KEY_NUM):
    for lbl,clr,nm in [(0,PALETTE["neg"],"Low Risk"),(1,PALETTE["pos"],"High Risk")]:
        axes[i].hist(df_sel[df_sel["target"]==lbl][col],bins=30,alpha=0.65,
                     color=clr,label=nm,edgecolor="white")
    axes[i].set_title(col.replace("_"," ").title(),fontweight="bold")
    axes[i].legend(fontsize=8)
for j in range(len(KEY_NUM),len(axes)): axes[j].set_visible(False)
plt.suptitle("Feature Distributions by Risk Class",fontsize=13,fontweight="bold")
plt.tight_layout(); plt.show()

## 10. Train / Val / Test split

>Split before SMOTE — prevents data leakage.

In [ ]:
FEATURE_COLS = [c for c in df_sel.columns if c!="target"]
X_all = df_sel[FEATURE_COLS].values.astype(np.float32)
y_all = df_sel["target"].values

X_train,X_temp,y_train,y_temp = train_test_split(
    X_all,y_all,test_size=0.30,random_state=RANDOM_SEED,stratify=y_all)
X_val,X_test,y_val,y_test = train_test_split(
    X_temp,y_temp,test_size=0.50,random_state=RANDOM_SEED,stratify=y_temp)

for nm,X_,y_ in [("Train",X_train,y_train),("Val",X_val,y_val),("Test",X_test,y_test)]:
    print(f"{nm:5s}: {len(X_):>7,}  pos={y_.mean():.1%}")

## 11. SMOTE — train only

In [ ]:
before = np.bincount(y_train)
smote  = SMOTE(random_state=RANDOM_SEED)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
after  = np.bincount(y_train_bal)
print(f"Before: {before}  After: {after}")

## 12. Feature scaling

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)
print(f"Train shape: {X_train_scaled.shape}")

## 13. Save Phase 1 artefacts

In [ ]:
df_sel = df_sel.reset_index(drop=True)
df_sel.to_parquet("data/diabetes_clean.parquet", index=True)

np.save("data/X_train.npy", X_train_scaled)
np.save("data/X_val.npy", X_val_scaled)
np.save("data/X_test.npy", X_test_scaled)
np.save("data/y_train.npy", y_train_bal)
np.save("data/y_val.npy", y_val)
np.save("data/y_test.npy", y_test)

with open("data/feature_names.json","w") as f: json.dump(FEATURE_COLS,f,indent=2)
with open("data/scaler.pkl","wb") as f: pickle.dump(scaler,f)

for fname in ["diabetes_clean.parquet","X_train.npy","X_val.npy","X_test.npy",
              "y_train.npy","y_val.npy","y_test.npy","feature_names.json","scaler.pkl"]:
    shutil.copy2(f"data/{fname}", f"{DRIVE_DIR}/{fname}")

print("Phase 1 saved and synced to Drive.")